# Multi-layer animated reports

Returns a notebook-native HTML snapshot for Bronze, Silver/Gold, and ML.

In [ ]:
import html
from datetime import datetime, timezone
CATALOG='aidp_poc'
def run(sql): return spark.sql(sql).collect()
def tab(items,cols):
    h=''.join('<th>'+label+'</th>' for _,label in cols)
    b=''.join('<tr>'+''.join('<td>'+html.escape(str(getattr(x,key,None) if getattr(x,key,None) is not None else '—'))+'</td>' for key,_ in cols)+'</tr>' for x in items) or '<tr><td colspan="9">Waiting for data</td></tr>'
    return '<table><thead><tr>'+h+'</tr></thead><tbody>'+b+'</tbody></table>'
bronze=run(f'''SELECT COUNT(*) events,COUNT(DISTINCT stream_partition) partitions,MAX(ingested_at) latest_ingested FROM {CATALOG}.bronze.meter_reading''')[0]
silver=run(f'''SELECT COUNT(*) readings,COUNT(DISTINCT meter_id) meters,COUNT(DISTINCT interval_start_utc) intervals,MAX(interval_start_utc) latest FROM {CATALOG}.silver.interval_reading''')[0]
gold=run(f'''SELECT ROUND(SUM(consumption_kwh),2) kwh,ROUND(MAX(demand_kw),2) demand,SUM(tamper_reading_count) tamper,SUM(outage_reading_count) outage FROM {CATALOG}.gold.meter_interval_usage WHERE interval_start_utc=(SELECT MAX(interval_start_utc) FROM {CATALOG}.gold.meter_interval_usage)''')[0]
models=run(f'''SELECT model_version,COUNT(*) predictions,ROUND(AVG(prediction_kwh),4) avg_kwh,ROUND(MAX(prediction_kwh),4) max_kwh,MAX(scored_at) scored_at FROM {CATALOG}.ml.meter_predictions GROUP BY model_version ORDER BY scored_at DESC''')
latest_events=run(f'''SELECT stream_partition,stream_offset,stream_key,ingested_at FROM {CATALOG}.bronze.meter_reading ORDER BY ingested_at DESC LIMIT 6''')
segments=run(f'''SELECT customer_segment,ROUND(SUM(consumption_kwh),2) kwh,ROUND(MAX(demand_kw),2) peak_kw FROM {CATALOG}.gold.meter_interval_usage WHERE interval_start_utc=(SELECT MAX(interval_start_utc) FROM {CATALOG}.gold.meter_interval_usage) GROUP BY customer_segment ORDER BY kwh DESC''')
preds=run(f'''SELECT meter_id,ROUND(prediction_kwh,4) prediction_kwh,interval_start_utc FROM {CATALOG}.ml.meter_predictions ORDER BY scored_at DESC LIMIT 8''')
stamp=datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
css='''<style>.mlr{font-family:Arial;color:#edf5ff;background:radial-gradient(circle at 5% 0,#334a98,#10162f 50%,#060914);padding:24px;border-radius:20px}.mlr h1{margin:0;font-size:28px}.sub{color:#b8c9ef;font-size:12px;margin:6px 0 18px}.report{background:linear-gradient(135deg,#1d2d61,#101a3a);border:1px solid #5b73b7;border-radius:16px;padding:17px;margin:14px 0;box-shadow:0 14px 26px #0005;animation:up .6s ease-out}.report:hover{transform:perspective(800px) rotateX(1deg) rotateY(-1deg)}.tag{display:inline-block;background:#74e9ff;color:#07101e;border-radius:14px;padding:5px 9px;font-size:11px;font-weight:bold}.grid{display:grid;grid-template-columns:repeat(4,1fr);gap:11px;margin:13px 0}.card{background:#0c1530;border:1px solid #455d9d;border-radius:12px;padding:13px}.label{font-size:10px;letter-spacing:1px;color:#a9bee8}.value{font-size:24px;font-weight:bold;color:#7feaff;margin-top:6px}table{width:100%;border-collapse:collapse}th{text-align:left;color:#aac0ed;font-size:10px;padding:8px 4px}td{padding:9px 4px;border-top:1px solid #ffffff1c}td:not(:first-child),th:not(:first-child){text-align:right}@keyframes up{from{opacity:.2;transform:translateY(16px)}}@media(max-width:700px){.grid{grid-template-columns:repeat(2,1fr)}}</style>'''
page=css+f'''<div class="mlr"><h1>⚡ GridPulse Layer Intelligence</h1><div class="sub">Animated streaming snapshot · {stamp} · rerun this cell to refresh data</div><div class="report"><span class="tag">01 BRONZE · RAW EVENT LANDING</span><div class="grid"><div class="card"><div class="label">EVENTS RECEIVED</div><div class="value">{bronze.events:,}</div></div><div class="card"><div class="label">PARTITIONS</div><div class="value">{bronze.partitions:,}</div></div><div class="card"><div class="label">LATEST INGEST</div><div class="value">{html.escape(str(bronze.latest_ingested))}</div></div><div class="card"><div class="label">FORMAT</div><div class="value">RAW JSON</div></div></div><b>Latest stream events</b>{tab(latest_events,[('stream_partition','Partition'),('stream_offset','Offset'),('stream_key','Meter'),('ingested_at','Ingested')])}</div><div class="report"><span class="tag">02 SILVER + GOLD · VALIDATED ENERGY</span><div class="grid"><div class="card"><div class="label">VALID READINGS</div><div class="value">{silver.readings:,}</div></div><div class="card"><div class="label">METERS</div><div class="value">{silver.meters:,}</div></div><div class="card"><div class="label">INTERVALS</div><div class="value">{silver.intervals:,}</div></div><div class="card"><div class="label">LATEST ENERGY</div><div class="value">{gold.kwh or 0} kWh</div></div></div><b>Current usage by segment</b>{tab(segments,[('customer_segment','Segment'),('kwh','Energy kWh'),('peak_kw','Peak kW')])}</div><div class="report"><span class="tag">03 ML · NEXT-INTERVAL FORECAST</span><div class="grid"><div class="card"><div class="label">MODEL VERSION</div><div class="value">{html.escape(str(models[0].model_version if models else '—'))}</div></div><div class="card"><div class="label">PREDICTIONS</div><div class="value">{models[0].predictions if models else 0:,}</div></div><div class="card"><div class="label">AVERAGE FORECAST</div><div class="value">{models[0].avg_kwh if models else 0} kWh</div></div><div class="card"><div class="label">MAXIMUM FORECAST</div><div class="value">{models[0].max_kwh if models else 0} kWh</div></div></div><b>Latest meter forecasts</b>{tab(preds,[('meter_id','Meter'),('prediction_kwh','Forecast kWh'),('interval_start_utc','Interval')])}</div></div>'''
class InlineReport:
    def __init__(self,markup): self.markup=markup
    def _repr_html_(self): return self.markup
    def __repr__(self): return 'HTML renderer unavailable.'
InlineReport(page)